## 提取 Markdown 标题为纯文本

递归扫描 `knowledgeBase/md_to_chunk/` 下所有 `.md` 文件，提取 ATX 标题后依次：

1. **去掉括号及其中内容**：全角 `（…）` 与半角 `(...)`（不含嵌套括号时一次即可；多处会循环剥除）。
2. **按词元切行**：在**空格**（含全角空格 `\u3000`）与**顿号 `、`** 处切分，使每个词元单独一行（例：`卷二 总释下` → `卷二` / `总释下`）。
3. **去重**：全库范围内保留**首次出现**顺序。

**运行方式**：在仓库根目录打开并运行；必要时修改 `ROOT` 绝对路径。

**依赖**：Python 标准库（`pathlib`、`re`）。

In [2]:
from pathlib import Path
import re

# 默认：仓库根目录为当前工作目录
ROOT = Path.cwd() / "knowledgeBase" / "md_to_chunk"
if not ROOT.is_dir():
    ROOT = Path(r"d:\postgraduate_study\graduation_thesis\llm_rag\myArAppRag\knowledgeBase\md_to_chunk")

# ATX 标题：行首最多 3 个空格 + 1～6 个 # + 空格 + 正文（与 CommonMark 常见写法一致）
ATX_HEADING = re.compile(r"^\s{0,3}(#{1,6})(?:\s+(.*))?$")
PAREN_FULL = re.compile(r"（[^）]*）")
PAREN_HALF = re.compile(r"\([^()]*\)")
# 空格（ASCII / 全角）与顿号处切分为词元
TOKEN_SPLIT = re.compile(r"[\s\u3000、]+")


def line_to_heading_plaintext(line: str) -> str | None:
    line = line.rstrip("\n\r")
    m = ATX_HEADING.match(line)
    if not m:
        return None
    rest = (m.group(2) or "").strip()
    if not rest:
        return None
    rest = re.sub(r"\s+#+\s*$", "", rest).strip()
    return rest or None


def strip_parens(s: str) -> str:
    """删除全角/半角括号及其中的内容（反复直到稳定，以处理紧邻的多段）。"""
    prev = None
    while prev != s:
        prev = s
        s = PAREN_FULL.sub("", s)
        s = PAREN_HALF.sub("", s)
    return s


def split_heading_to_tokens(heading: str) -> list[str]:
    """去括号后按空格、顿号切分，每段为一词元。"""
    s = strip_parens(heading).strip()
    if not s:
        return []
    parts = TOKEN_SPLIT.split(s)
    return [p.strip() for p in parts if p.strip()]


def dedupe_preserve_order(items: list[str]) -> list[str]:
    seen: set[str] = set()
    out: list[str] = []
    for x in items:
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out


def collect_headings(root: Path) -> list[str]:
    titles: list[str] = []
    for md in sorted(root.rglob("*.md")):
        try:
            text = md.read_text(encoding="utf-8")
        except UnicodeDecodeError:
            text = md.read_text(encoding="utf-8-sig")
        for line in text.splitlines():
            t = line_to_heading_plaintext(line)
            if t is not None:
                titles.append(t)
    return titles


titles = collect_headings(ROOT)
all_tokens: list[str] = []
for t in titles:
    all_tokens.extend(split_heading_to_tokens(t))
unique_tokens = dedupe_preserve_order(all_tokens)

print(f"目录: {ROOT.resolve()}")
print(f"原始标题行数: {len(titles)}，切分后词元数: {len(all_tokens)}，去重后: {len(unique_tokens)}\n")
for line in unique_tokens:
    print(line)

目录: D:\postgraduate_study\graduation_thesis\llm_rag\myArAppRag\knowledgeBase\md_to_chunk
原始标题行数: 1034，切分后词元数: 904，去重后: 529

引言
《营造法式》注释序
《营造法式》
《营造法式》的编修
李诫
八百余年来《营造法式》的版本
我们这一次的整理
注释工作
我们在注释中遇到的一些问题
甲
文字方面的问题。
乙
图样方面的问题。
我们整理工作的总原则
进新修《营造法式》序
札子
[^1]
《营造法式》看详
方圆平直
取径围
定功
取正
定平
墙
举折
诸作异名
总诸作看详
卷一
总释上
宫
阙
殿
堂附
楼
亭
台榭
城
柱础
材
栱
飞昂
爵头
枓
铺作
平坐
梁
柱
阳马
侏儒柱
斜柱
卷二
总释下
栋
两际
搏风
柎
椽
檐
余廉切，或作櫩，俗作檐者非是。
门
乌头门
华表
窗
平棋
斗八藻井
勾栏
拒马叉子
屏风
槏柱
露篱
鸱尾
瓦
涂
彩画
阶
砖
井
总例
卷三
壕寨
石作制度
壕寨制度
立基
筑基
筑临水基
造作次序
角石
角柱
殿阶基
压栏石
地面石
殿阶螭首
螭首
殿内斗八
斗八
踏道
重台勾栏
单勾栏
螭子石
门砧限
门砧
地栿
流杯渠
坛
卷輂水窗
水槽子
马台
井口石
山棚
幡竿颊
赑屃鳌坐碑
笏头碣
卷四
大木作制度一
其名有三：一曰章，二曰材，三曰方桁。
总铺作次序
卷五
大木作制度二
阑额
其名有二：一曰楹，二曰柱。
搏风板
其名有二：一曰荣，二曰搏风
其名有三：一曰柎，二曰复栋，三曰替木。
卷六
小木作制度一
板门双扇板门
独扇板门
板门
软门牙头护缝软门
合板软门
破子棂窗
睒电窗
板棂窗
截间板帐
照壁屏风骨
隔截横钤立旌
板引檐
水槽
井屋子
地棚
卷七
小木作制度二
格子门
栏槛勾窗
殿内截间格子
堂阁内截间格子
殿阁照壁板
障日板
廊屋照壁板
胡梯
垂鱼
惹草
栱眼壁板
裹栿板
擗帘竿
护殿阁檐竹网木贴
卷八
小木作制度三
小斗八藻井
叉子
棵笼子
井亭子
牌
卷九
小木作制度四
佛道帐
卷十
小木作制度五
牙脚帐
九脊小帐
壁帐
卷十一
小木作制度六
转轮经藏
壁藏
卷十二雕作
旋作
锯作
竹作制度
雕作制度
混作
雕插写生花
起突卷叶花
剔地洼叶花
旋作制度
殿堂等杂用名件
照壁板宝床上名件
佛道帐上名件
锯

### 可选：写入文本文件

取消注释下面单元中的写入代码，可将**去重后的词元**保存为 `md_headings_tokens.txt`（路径可自行修改）。

In [ ]:
OUT_FILE = Path.cwd() / "knowledgeBase" / "md_to_chunk" / "md_headings_tokens.txt"
OUT_FILE.write_text("\n".join(unique_tokens) + "\n", encoding="utf-8")
print("已写入:", OUT_FILE.resolve())

In [3]:
# 清洗 knowledgeBase/pdfParse/cleaned_data/termsName_forPrompt.text
# 1) 删除 strip 后仅一个字的行  2) 去重（保留首次出现顺序）  3) 写回原文件（运行前可自行备份）
from pathlib import Path

TERMS_FILE = Path.cwd() / "knowledgeBase" / "pdfParse" / "cleaned_data" / "termsName_forPrompt.text"
if not TERMS_FILE.is_file():
    TERMS_FILE = Path(
        r"d:\postgraduate_study\graduation_thesis\llm_rag\myArAppRag\knowledgeBase\pdfParse\cleaned_data\termsName_forPrompt.text"
    )


def _dedupe_preserve_order(items: list[str]) -> list[str]:
    seen: set[str] = set()
    out: list[str] = []
    for x in items:
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out


raw_lines = TERMS_FILE.read_text(encoding="utf-8").splitlines()
kept: list[str] = []
removed_single = 0
for line in raw_lines:
    s = line.strip()
    if not s:
        continue
    if len(s) == 1:
        removed_single += 1
        continue
    kept.append(s)

before_dedupe = len(kept)
terms_cleaned = _dedupe_preserve_order(kept)
removed_dup = before_dedupe - len(terms_cleaned)

print(f"文件: {TERMS_FILE.resolve()}")
print(
    f"去掉单字行: {removed_single}，去重前: {before_dedupe}，去重后: {len(terms_cleaned)}（重复条目 {removed_dup}）"
)

TERMS_FILE.write_text("\n".join(terms_cleaned) + "\n", encoding="utf-8")
print("已写回原文件。")

文件: D:\postgraduate_study\graduation_thesis\llm_rag\myArAppRag\knowledgeBase\pdfParse\cleaned_data\termsName_forPrompt.text
去掉单字行: 62，去重前: 1190，去重后: 1087（重复条目 103）
已写回原文件。
